In [ ]:
!pip install maldideepkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
except ImportError:
    print('Running locally -- skipping Google Drive mount')

# Drug-Wide Logistic Regression Screen — PCA + L2 + Threshold Tuning

This notebook runs the full **PCA + L2-regularised logistic regression +
threshold tuning** pipeline (Approach C from `0-0-2-LogisticAnalysis`) on
**every drug** with sufficient data in the DRIAMS A2018 cohort (all species).

The aim is to map the linear-performance landscape across all antimicrobial
agents, rather than focusing on a single drug at a time.

---
## Filters

| Criterion | Threshold |
|---|---|
| Min total tested samples | >= 400 |
| Species | All (no filter) |
| Min per class in train | >= 10 R and >= 10 S |

---
## Pipeline (per drug)

1. Extract spectra + labels via `prepare_drug_data()`
2. Leak-safe `log1p + standardise` preprocessing
3. `StandardScaler` + `PCA(0.94)` on train only
4. `GridSearchCV(L2 LR, C_grid, cv=3)` on PCA-reduced train
5. Threshold tuning on validation split (maximise BalAcc)
6. Evaluate on test split — record all metrics

In [ ]:
# =============================================================================
# 1. IMPORTS & CONFIGURATION
# =============================================================================

# --- Standard library ---
import warnings          # suppress FutureWarning noise from sklearn/pandas
from pathlib import Path # cross-platform path handling

# --- Numerical & data manipulation ---
import numpy as np       # matrix operations, .npy loading
import pandas as pd      # CSV metadata, pivot tables

# --- Plotting ---
import matplotlib.pyplot as plt  # bar charts, ROC curves, confusion matrices
import matplotlib.ticker as mticker

# --- MaldiDeepKit: preprocessing layer ---
from maldideepkit.base.data import (
    fit_input_transform,    # compute per-bin mean/std from training split only
    apply_input_transform,  # apply log1p / standardise / robust to any split
)

# --- Scikit-learn: classifiers & preprocessing ---
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- Scikit-learn: model selection ---
from sklearn.model_selection import GridSearchCV

# --- Scikit-learn: metrics ---
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)

# Suppress excessive warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility
SEED = 42
np.random.seed(SEED)


# --- Paths ---
# Local (uncomment if running on your machine)
DATA_DIR = Path("../../A2018/A2018")
OUT_DIR  = Path("./results_drug_screen")

# Colab (uncomment if running on Google Colab)
# DATA_DIR = Path("/content/drive/MyDrive/Flower/Duroux_D_dirams/A2018/A2018")
# OUT_DIR  = Path("/content/drive/MyDrive/Flower/Duroux_D_dirams/0-Analysis/0-0-2-1-LogisticAnalysis/results_drug_screen")

SPECTRA_PATH   = DATA_DIR / "rawSpectra_data.npy"
SPLITS_PATH    = DATA_DIR / "data_splits.csv"
LONG_TABLE_PATH = DATA_DIR / "combined_long_table.csv"

OUT_DIR.mkdir(exist_ok=True)

print(f"Data dir:  {DATA_DIR.resolve()}")
print(f"Output to: {OUT_DIR.resolve()}")

---
## 2. LOAD SPECTRA

In [ ]:
# -----------------------------------------------------------------------------
# 2.1  Load the binned spectra matrix
# -----------------------------------------------------------------------------

X_full = np.load(SPECTRA_PATH).astype("float32")
n_samples, n_bins = X_full.shape
mz_axis = np.arange(2000, 2000 + n_bins * 3, 3)

print(f"Spectra matrix: {X_full.shape}")
print(f"  Samples       {n_samples}")
print(f"  Bins          {n_bins}  (3 Da from {mz_axis[0]} to {mz_axis[-1]} Da)")
print(f"  dtype         {X_full.dtype}")

---
## 3. LOAD METADATA

In [ ]:
# -----------------------------------------------------------------------------
# 3.1  Load split assignments and the long-table
# -----------------------------------------------------------------------------

splits_df = pd.read_csv(SPLITS_PATH)
long_df   = pd.read_csv(LONG_TABLE_PATH)

print("Split distribution:")
print(splits_df["Set"].value_counts().to_string())
print(f"\nMetadata: {long_df.shape[0]:,} rows, "
      f"{long_df['species'].nunique()} species, {long_df['drug'].nunique()} drugs")

# -----------------------------------------------------------------------------
# 3.2  Pivot the long table -> sample x drug matrix
# -----------------------------------------------------------------------------

pivot_df = long_df.pivot_table(
    index="sample_id", columns="drug", values="response"
)
pivot_df = pivot_df.reindex(splits_df["sample_id"])

print(f"\nPivot shape: {pivot_df.shape}")
print(f"\nAll drugs sorted by number of tested samples:")
drug_counts = pivot_df.notna().sum().sort_values(ascending=False)
for drug, n in drug_counts.items():
    flag = "  <- qualifying" if n >= 400 else ""
    print(f"  {drug:22s}  n={n:5d}{flag}")

---
## 4. PER-DRUG SCREEN — PCA + L2 LR + THRESHOLD TUNING

### Filters applied
- Total tested samples >= **400**
- Train split has >= **10** resistant AND >= **10** susceptible samples

Drugs failing either criterion are skipped with a clear message.

In [ ]:
# =============================================================================
# 4.1  Helper: extract X, y and split masks for a given drug
# =============================================================================

def prepare_drug_data(drug_name):
    '''Extract spectra and labels for samples tested against *drug_name*.'''
    valid_mask = pivot_df[drug_name].notna().to_numpy()
    X = X_full[valid_mask]
    y = pivot_df[drug_name].dropna().to_numpy(dtype=np.int64)
    split_subset = splits_df.loc[valid_mask, "Set"].to_numpy()

    masks = {
        "train": (split_subset == "train"),
        "test":  (split_subset == "test"),
        "valid": (split_subset == "validation"),
    }
    return X, y, masks


# =============================================================================
# 4.2  Configure grid-search parameters (same as 0-0-2)
# =============================================================================

C_grid = np.linspace(1e-6, 0.1, 20)
THRESHOLDS = np.linspace(0.05, 0.95, 91)

# =============================================================================
# 4.3  Screen all qualifying drugs
# =============================================================================

# Build the list of drugs with >= 400 tested samples
qualifying_drugs = drug_counts[drug_counts >= 400].index.tolist()
print(f"Qualifying drugs (n >= 400): {len(qualifying_drugs)}")
print(f"  {qualifying_drugs}")
print()

results = []
skipped = []

for idx, drug in enumerate(qualifying_drugs):
    print(f"[{idx+1:2d}/{len(qualifying_drugs)}] {drug}")

    # --- Prepare data ---
    X_d, y_d, masks_d = prepare_drug_data(drug)

    X_tr = X_d[masks_d["train"]]
    y_tr = y_d[masks_d["train"]]
    X_te = X_d[masks_d["test"]]
    y_te = y_d[masks_d["test"]]
    X_va = X_d[masks_d["valid"]]
    y_va = y_d[masks_d["valid"]]

    n_train = len(y_tr)
    n_test  = len(y_te)
    r_train = (y_tr == 0).sum()
    s_train = (y_tr == 1).sum()
    r_test  = (y_te == 0).sum()
    s_test  = (y_te == 1).sum()

    # --- Class-balance filter ---
    if r_train < 10 or s_train < 10:
        skipped.append({
            "drug": drug,
            "reason": f"train imbalance: R={r_train} S={s_train} (need >= 10 each)",
            "n_train": n_train,
        })
        print(f"  SKIP: R={r_train} S={s_train} in train (need >= 10 each)")
        print()
        continue

    # --- Leak-safe preprocessing ---
    state = fit_input_transform(X_tr, "log1p+standardize")
    X_tr = apply_input_transform(X_tr, state)
    X_te = apply_input_transform(X_te, state)
    X_va = apply_input_transform(X_va, state)

    # --- PCA on train only ---
    scaler = StandardScaler().fit(X_tr)
    pca = PCA(n_components=0.94, random_state=SEED).fit(scaler.transform(X_tr))
    X_tr_pca = pca.transform(scaler.transform(X_tr))
    X_te_pca = pca.transform(scaler.transform(X_te))
    X_va_pca = pca.transform(scaler.transform(X_va))

    # --- Grid-search L2 strength C on PCA train ---
    grid = GridSearchCV(
        LogisticRegression(
            penalty="l2", solver="lbfgs", class_weight="balanced",
            max_iter=5000, random_state=SEED
        ),
        param_grid={"C": C_grid},
        cv=3,
        scoring="balanced_accuracy",
        n_jobs=1,
        verbose=0,
    )
    grid.fit(X_tr_pca, y_tr)
    lr_model = grid.best_estimator_

    # --- Threshold tuning on validation split ---
    proba_va = lr_model.predict_proba(X_va_pca)[:, 1]
    balaccs_va = [balanced_accuracy_score(y_va, proba_va >= t) for t in THRESHOLDS]
    best_thresh = THRESHOLDS[np.argmax(balaccs_va)]

    # --- Evaluate on test split ---
    proba_te = lr_model.predict_proba(X_te_pca)[:, 1]
    preds_te = (proba_te >= best_thresh).astype(np.int64)

    test_balacc = balanced_accuracy_score(y_te, preds_te)
    test_auc    = roc_auc_score(y_te, proba_te)

    train_preds = lr_model.predict(X_tr_pca)
    train_balacc = balanced_accuracy_score(y_tr, train_preds)
    gap = train_balacc - test_balacc

    results.append({
        "drug": drug,
        "n_total": n_train + n_test + len(y_va),
        "n_train": n_train,
        "n_test":  n_test,
        "%R (train)":  f"{r_train / n_train * 100:.1f}",
        "%R (test)":   f"{r_test  / n_test  * 100:.1f}",
        "n_PCA": pca.n_components_,
        "best_C": grid.best_params_["C"],
        "cv_balacc": grid.best_score_,
        "best_thresh": best_thresh,
        "train_balacc": train_balacc,
        "test_balacc": test_balacc,
        "test_auc": test_auc,
        "gap": gap,
    })

    print(f"  n={n_train+n_test+len(y_va):4d}  "
          f"PCA={pca.n_components_:4d}  C={grid.best_params_['C']:.2e}  "
          f"thresh={best_thresh:.2f}  train_balacc={train_balacc:.3f}  "
          f"test_balacc={test_balacc:.3f}  test_auc={test_auc:.3f}")
    print()

# --- Summary ---
print(f"\nDone. {len(results)} drugs processed, {len(skipped)} skipped.")
if skipped:
    print("Skipped drugs:")
    for s in skipped:
        print(f"  {s['drug']:22s}  {s['reason']} (train n={s['n_train']})")

---
## 5. RESULTS

In [ ]:
# =============================================================================
# 5.1  Build results DataFrame (ranked by test BalAcc)
# =============================================================================

df_results = pd.DataFrame(results).sort_values("test_balacc", ascending=False).reset_index(drop=True)
df_results.index = df_results.index + 1  # 1-based rank
df_results.index.name = "Rank"

# Display
cols_show = ["drug", "n_train", "n_test", "n_PCA", "best_C", "cv_balacc",
             "train_balacc", "test_balacc", "test_auc", "gap", "best_thresh"]
print(f"Drug ranking by test Balanced Accuracy ({len(df_results)} drugs):")
print()
print(df_results[cols_show].to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

# Save CSV
csv_path = OUT_DIR / "drug_summary.csv"
df_results.to_csv(csv_path)
print(f"\nSaved to: {csv_path.resolve()}")

# =============================================================================
# 5.2  Horizontal bar chart — ranked by test BalAcc
# =============================================================================

fig, ax = plt.subplots(figsize=(10, max(5, len(df_results) * 0.35)))

drugs_rev = df_results["drug"].values[::-1]
balacc_rev = df_results["test_balacc"].values[::-1]
auc_rev    = df_results["test_auc"].values[::-1]

y_pos = np.arange(len(drugs_rev))
height = 0.35

bars_bal = ax.barh(y_pos + height/2, balacc_rev, height, label="Test BalAcc", color="#1f77b4")
bars_auc = ax.barh(y_pos - height/2, auc_rev,    height, label="Test AUC",   color="#ff7f0e")

ax.set_yticks(y_pos)
ax.set_yticklabels(drugs_rev)
ax.set_xlabel("Score")
ax.set_title(f"PCA + L2 LR — Drug-Wide Screen ({len(df_results)} drugs)")
ax.legend(loc="lower right")
ax.set_xlim(0, 1.05)
ax.axvline(0.5, color="gray", ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / "drug_ranking.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# 5.3  Histogram of per-drug test AUCs
# =============================================================================

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_results["test_auc"], bins=min(12, len(df_results)),
        color="#1f77b4", edgecolor="white", alpha=0.8)
ax.axvline(0.5, color="gray", ls="--", alpha=0.5, label="Random (0.5)")
ax.axvline(df_results["test_auc"].median(), color="#d62728", ls="--",
           label=f"Median = {df_results['test_auc'].median():.3f}")
ax.set_xlabel("Test AUC")
ax.set_ylabel("Number of drugs")
ax.set_title("Distribution of Per-Drug Test AUC")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "drug_auc_histogram.pdf", bbox_inches="tight")
plt.show()

# Key stats
print("Per-drug test AUC statistics:")
print(f"  Median:  {df_results['test_auc'].median():.4f}")
print(f"  Mean:    {df_results['test_auc'].mean():.4f}")
print(f"  Min:     {df_results['test_auc'].min():.4f}  ({df_results.loc[df_results['test_auc'].idxmin(), 'drug']})")
print(f"  Max:     {df_results['test_auc'].max():.4f}  ({df_results.loc[df_results['test_auc'].idxmax(), 'drug']})")

n_gt_70 = (df_results["test_auc"] > 0.70).sum()
n_gt_75 = (df_results["test_auc"] > 0.75).sum()
n_gt_80 = (df_results["test_auc"] > 0.80).sum()
print(f"\nDrugs with test AUC > 0.70:  {n_gt_70}")
print(f"Drugs with test AUC > 0.75:  {n_gt_75}")
print(f"Drugs with test AUC > 0.80:  {n_gt_80}")

---
## 6. REPORT

In [ ]:
# =============================================================================
# 6.1  Generate report.md
# =============================================================================

report_path = Path("report.md")

top5 = df_results.head(5)
bottom5 = df_results.tail(5)

report_lines = [
    "# Drug-Wide Logistic Regression Screen — Report",
    "",
    f"**Dataset:** DRIAMS A2018 cohort, all species",
    f"**Preprocessing:** log1p + standardise (leak-safe)",
    f"**Pipeline:** PCA (94% variance) + L2 LR + GridSearchCV(C) + threshold tuning",
    "",
    f"**Qualifying drugs:** {len(qualifying_drugs)} with >= 400 tested samples",
    f"**Processed:** {len(results)} drugs",
    f"**Skipped:** {len(skipped)} (class imbalance < 10 per class in train)",
    "",
    "---",
    "",
    "## Top-5 Drugs by Test Balanced Accuracy",
    "",
    "| Rank | Drug | n | PCA | Test BalAcc | Test AUC | Train BalAcc | Gap |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for _, r in top5.iterrows():
    report_lines.append(
        f"| {r.name} | {r['drug']} | {r['n_test']} | {r['n_PCA']} | "
        f"{r['test_balacc']:.4f} | {r['test_auc']:.4f} | {r['train_balacc']:.4f} | "
        f"{r['gap']:+.4f} |"
    )

report_lines += [
    "",
    "## Bottom-5 Drugs by Test Balanced Accuracy",
    "",
    "| Rank | Drug | n | PCA | Test BalAcc | Test AUC | Train BalAcc | Gap |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for _, r in bottom5.iterrows():
    report_lines.append(
        f"| {r.name} | {r['drug']} | {r['n_test']} | {r['n_PCA']} | "
        f"{r['test_balacc']:.4f} | {r['test_auc']:.4f} | {r['train_balacc']:.4f} | "
        f"{r['gap']:+.4f} |"
    )

report_lines += [
    "",
    "## AUC Statistics",
    "",
    f"| Statistic | Value |",
    f"|---|---|",
    f"| Median | {df_results['test_auc'].median():.4f} |",
    f"| Mean   | {df_results['test_auc'].mean():.4f} |",
    f"| Min    | {df_results['test_auc'].min():.4f} |",
    f"| Max    | {df_results['test_auc'].max():.4f} |",
    f"| > 0.70 | {n_gt_70} drugs |",
    f"| > 0.75 | {n_gt_75} drugs |",
    f"| > 0.80 | {n_gt_80} drugs |",
    "",
    "---",
    "",
    "## Full Results",
    "",
    f"See `{OUT_DIR}/drug_summary.csv` for all metrics.",
    "",
    "## Generated Figures",
    "",
    f"- `{OUT_DIR}/drug_ranking.pdf` — horizontal bar chart, ranked by test BalAcc",
    f"- `{OUT_DIR}/drug_auc_histogram.pdf` — distribution of per-drug test AUCs",
    "",
    "---",
    "",
    "## Skipped Drugs",
    "",
]

if skipped:
    for s in skipped:
        report_lines.append(f"- **{s['drug']}**: {s['reason']}")
else:
    report_lines.append("- None (all qualifying drugs passed the filters)")

report_lines += [
    "",
    "---",
    "",
    "## Configuration",
    "",
    "| Parameter | Value |",
    "|---|---|",
    "| Model | sklearn `LogisticRegression` (L2, balanced, LBFGS, max_iter=5000) |",
    "| Dimensionality reduction | PCA retaining 94% variance |",
    "| C grid | linspace(1e-6, 0.1, 20) |",
    "| CV folds | 3 (GridSearchCV, scoring=balanced_accuracy) |",
    "| Threshold sweep | 0.05 – 0.95 (91 steps, optimising BalAcc on validation split) |",
    "| Random seed | 42 |",
    "",
    f"*Report generated by `0-0-2-1-LogisticAnalysis.ipynb`*",
]

with open(report_path, "w") as f:
    f.write("\n".join(report_lines) + "\n")

print(f"Report saved to: {report_path.resolve()}")
print(report_path.read_text())